# Cognizant India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** careers.cognizant.com/india-en/jobs (RSS + Selenium)

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_CODE   = ""  # e.g. "in" for SmartRecruiters country= param; "" = all
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-03-31 23:07:49
Location filter: '' (empty = broad/global scraping)


In [3]:
COMPANY = "Cognizant"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Cognizant/Outputs/2026_03_31


In [4]:
print("=" * 60)
print("COGNIZANT INDIA JOB SCRAPER")
print("Source: careers.cognizant.com/india-en/jobs (XML feed)")
print("=" * 60)

cognizant_jobs = []
session = get_session()

# XML feed — uses <job> elements (NOT <item> like standard RSS)
try:
    xml_url = "https://careers.cognizant.com/india-en/jobs/xml/?rss=true"
    print(f"  Fetching XML feed: {xml_url}")
    resp = session.get(xml_url, timeout=30)
    if resp.status_code == 200:
        soup = BeautifulSoup(resp.text, "xml")
        # Cognizant XML uses <job> tags, not <item>
        jobs_xml = soup.find_all("job")
        if not jobs_xml:
            # Fallback: try <item> in case format changed
            jobs_xml = soup.find_all("item")
        print(f"  XML feed returned {len(jobs_xml)} job entries")

        for job_el in jobs_xml:
            title = job_el.find("title")
            title = title.get_text(strip=True) if title else ""
            url_el = job_el.find("url")
            link = url_el.get_text(strip=True) if url_el else ""
            if not link:
                link_el = job_el.find("link")
                link = link_el.get_text(strip=True) if link_el else ""
            desc = job_el.find("description")
            desc = desc.get_text(strip=True) if desc else ""
            date_el = job_el.find("date")
            pub_date = date_el.get_text(strip=True) if date_el else ""
            req_id = job_el.find("requisitionid")
            req_id = req_id.get_text(strip=True) if req_id else ""
            city_el = job_el.find("city")
            city = city_el.get_text(strip=True) if city_el else ""
            country_el = job_el.find("country")
            country = country_el.get_text(strip=True) if country_el else ""
            category_el = job_el.find("category")
            category = category_el.get_text(strip=True) if category_el else ""
            remote_el = job_el.find("remotetype")
            remote_type = remote_el.get_text(strip=True) if remote_el else ""

            # Filter: only India jobs
            if country.lower() != "india":
                continue

            if title and is_valid_job_title(title):
                cognizant_jobs.append({
                    "job_id": req_id or (link.split("/")[-1] if link else str(len(cognizant_jobs))),
                    "title": title,
                    "company_name": "Cognizant",
                    "raw_jd_text": html_to_text(desc),
                    "location_city": city if city else "India",
                    "industry": "IT Services & Consulting",
                    "date_posted": pub_date[:10] if pub_date and len(pub_date) >= 10 else datetime.now().strftime("%Y-%m-%d"),
                    "is_active": True,
                    "job_url": link,
                    "business_unit": category,
                    "work_mode": "hybrid" if "hybrid" in remote_type.lower() else ("remote" if "remote" in remote_type.lower() else "onsite"),
                    "source_platform": "Cognizant XML Feed",
                })
        print(f"  India jobs after filtering: {len(cognizant_jobs)}")
    else:
        print(f"  XML feed returned HTTP {resp.status_code}")
except Exception as e:
    print(f"  XML feed failed: {e}")

# Selenium fallback if XML returned too few
if len(cognizant_jobs) < 5:
    print("  Trying Selenium on careers.cognizant.com...")
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC

    driver = setup_selenium()
    try:
        driver.get("https://careers.cognizant.com/india-en/jobs/")
        time.sleep(10)

        # Wait for job cards to render
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "[class*=\'job\'], [class*=\'search-result\'], [data-ph-at-id]"))
            )
        except:
            time.sleep(5)

        soup = BeautifulSoup(driver.page_source, "lxml")
        # Try multiple selector strategies
        cards = soup.select("[data-ph-at-id*=\'job\'], [class*=\'job-card\'], [class*=\'search-result-item\'], li[class*=\'job\']")
        if not cards:
            cards = soup.select("a[href*=\'/job/\']")
            cards = [c.parent for c in cards if c.parent and c.parent.name != "nav"]

        for card in cards:
            title_el = card.select_one("h2, h3, h4, [class*=\'title\'], a[href*=\'/job/\']")
            title = title_el.get_text(strip=True) if title_el else ""
            loc_el = card.select_one("[class*=\'location\'], [class*=\'city\']")
            loc = loc_el.get_text(strip=True) if loc_el else "India"

            if is_valid_job_title(title) and title not in [j["title"] for j in cognizant_jobs]:
                link = card.select_one("a[href]")
                href = link.get("href", "") if link else ""
                cognizant_jobs.append({
                    "job_id": href.split("/")[-1] if href else str(len(cognizant_jobs)),
                    "title": title,
                    "company_name": "Cognizant",
                    "raw_jd_text": card.get_text(" ", strip=True),
                    "location_city": loc.split(",")[0].strip(),
                    "industry": "IT Services & Consulting",
                    "date_posted": datetime.now().strftime("%Y-%m-%d"),
                    "is_active": True,
                    "job_url": href if href.startswith("http") else f"https://careers.cognizant.com{href}" if href else "",
                    "business_unit": "",
                    "source_platform": "Cognizant Selenium fallback",
                })
    except Exception as e:
        print(f"  Selenium error: {e}")
    finally:
        driver.quit()

print(f"Total Cognizant India jobs: {len(cognizant_jobs)}")


COGNIZANT INDIA JOB SCRAPER
Source: careers.cognizant.com/india-en/jobs (XML feed)
  Fetching XML feed: https://careers.cognizant.com/india-en/jobs/xml/?rss=true


  XML feed returned 2049 job entries


  India jobs after filtering: 518
Total Cognizant India jobs: 518


In [5]:
df_cognizant = save_results(cognizant_jobs, "Cognizant", OUTPUT_DIR)
if df_cognizant is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_cognizant.columns]
    print(df_cognizant[cols].head(10).to_string())


  [OK] Saved 455 jobs -> Cognizant_jobs_2026-03-31.csv
       Seniority: {'lead': 250, 'senior': 106, 'junior': 74, 'mid': 25}
       Work mode: {'hybrid': 423, 'onsite': 24, 'remote': 8}
       Has JD text: 454/455
       Has job URL: 455/455
       Has business unit: 455/455

Sample jobs:
                                                                       title location_city seniority_level             business_unit                                                                                                                     job_url
0                                                       SME-Policy Servicing         Noida          senior  Technology & Engineering                                              https://careers.cognizant.com/india-en/jobs/000673836031/sme-policy-servicing/
1                                                        PE-Policy Servicing         Noida          senior  Technology & Engineering                                               https://careers